In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("Airline_EDA") \
    .getOrCreate()

print(spark.version)

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
3,application_1783172556170_0004,pyspark3,idle,Link,Link,✔


SparkSession available as 'spark'.
2.4.0

In [3]:
df = spark.read.parquet(
    "s3a://airline-dataset-2020-2025/Silver/"
)

In [4]:
df.show(5)

+-------+-----+----------+---------+-------------------+-------------------------+---------------------------------------+------------------------+---------------------------+-------------------------------+---------------------------------------+----------------------------------------------+-------------------------------------------------+--------------------------------------------------+-----------------+------------------------+---------------------------+-----------+-------------------------------+---------------+------------------+------------------+------+--------------+-----------+---------------+---------------+---------+-------------+----------------+----------------+----+------------+---------+-------------+-------------+-------+----------+-------+--------+---------------+--------+--------------------+----------+-------+---------+--------+------+----------+-------+--------+---------------+--------+------------------+----------+---------+----------------+--------+----------

In [5]:
eda_df = df.select(
    "Marketing_Airline_Network",
    "Origin",
    "OriginState",
    "Dest",
    "DestState",
    "Month",
    "Year"
)

In [6]:
eda_df.show(10, False)

+-------------------------+------+-----------+----+---------+-----+----+
|Marketing_Airline_Network|Origin|OriginState|Dest|DestState|Month|Year|
+-------------------------+------+-----------+----+---------+-----+----+
|AS                       |ORD   |IL         |PDX |OR       |7    |2021|
|AS                       |LAS   |NV         |SEA |WA       |7    |2021|
|AS                       |SEA   |WA         |OMA |NE       |7    |2021|
|AS                       |OMA   |NE         |SEA |WA       |7    |2021|
|AS                       |SEA   |WA         |GEG |WA       |7    |2021|
|AS                       |SEA   |WA         |DEN |CO       |7    |2021|
|AS                       |MCI   |MO         |SEA |WA       |7    |2021|
|AS                       |PDX   |OR         |ORD |IL       |7    |2021|
|AS                       |SEA   |WA         |LAS |NV       |7    |2021|
|AS                       |PHX   |AZ         |SEA |WA       |7    |2021|
+-------------------------+------+-----------+----+

In [7]:
print("Total Rows :", eda_df.count())
print("Total Columns :", len(eda_df.columns))

Total Rows : 40910253
Total Columns : 7

In [8]:
eda_df.printSchema()


root
 |-- Marketing_Airline_Network: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DestState: string (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Year: integer (nullable = true)

In [9]:
print(eda_df.dtypes)

[('Marketing_Airline_Network', 'string'), ('Origin', 'string'), ('OriginState', 'string'), ('Dest', 'string'), ('DestState', 'string'), ('Month', 'int'), ('Year', 'int')]

In [10]:
eda_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in eda_df.columns
]).show()

+-------------------------+------+-----------+----+---------+-----+----+
|Marketing_Airline_Network|Origin|OriginState|Dest|DestState|Month|Year|
+-------------------------+------+-----------+----+---------+-----+----+
|                        0|     0|          0|   0|        0|    0|   0|
+-------------------------+------+-----------+----+---------+-----+----+

In [11]:
duplicates = eda_df.count() - eda_df.dropDuplicates().count()

print("Duplicate Records :", duplicates)

Duplicate Records : 40305290

In [12]:
for c in eda_df.columns:
    print(c, ":", eda_df.select(c).distinct().count())

Marketing_Airline_Network : 10
Origin : 390
OriginState : 53
Dest : 391
DestState : 53
Month : 12
Year : 6

In [13]:
eda_df.select("Marketing_Airline_Network") \
      .distinct() \
      .orderBy("Marketing_Airline_Network") \
      .show(100, False)

+-------------------------+
|Marketing_Airline_Network|
+-------------------------+
|AA                       |
|AS                       |
|B6                       |
|DL                       |
|F9                       |
|G4                       |
|HA                       |
|NK                       |
|UA                       |
|WN                       |
+-------------------------+

In [14]:
airline_summary = (
    eda_df.groupBy("Marketing_Airline_Network")
          .count()
          .orderBy(desc("count"))
)

airline_summary.show(100, False)

+-------------------------+--------+
|Marketing_Airline_Network|count   |
+-------------------------+--------+
|AA                       |10407897|
|DL                       |8526129 |
|WN                       |7582834 |
|UA                       |7427635 |
|AS                       |2238407 |
|B6                       |1366470 |
|NK                       |1278352 |
|F9                       |968030  |
|G4                       |694895  |
|HA                       |419604  |
+-------------------------+--------+

In [15]:
origin_summary = (
    eda_df.groupBy("Origin")
          .count()
          .orderBy(desc("count"))
)

origin_summary.show(25, False)

+------+-------+
|Origin|count  |
+------+-------+
|ATL   |1915120|
|ORD   |1778804|
|DFW   |1705528|
|DEN   |1683092|
|CLT   |1347556|
|LAX   |1086362|
|SEA   |1026212|
|PHX   |1021572|
|LAS   |999915 |
|IAH   |904228 |
|MCO   |859437 |
|LGA   |824195 |
|DTW   |771724 |
|EWR   |754423 |
|SFO   |744221 |
|BOS   |734638 |
|DCA   |734587 |
|MSP   |722032 |
|SLC   |661333 |
|JFK   |657831 |
|PHL   |651888 |
|MIA   |580098 |
|BNA   |528680 |
|BWI   |525425 |
|FLL   |505499 |
+------+-------+
only showing top 25 rows

In [16]:
print(
    "Distinct Origin Airports:",
    eda_df.select("Origin").distinct().count()
)

Distinct Origin Airports: 390

In [17]:
destination_summary = (
    eda_df.groupBy("Dest")
          .count()
          .orderBy(desc("count"))
)

destination_summary.show(25, False)

+----+-------+
|Dest|count  |
+----+-------+
|ATL |1914971|
|ORD |1778590|
|DFW |1705415|
|DEN |1682968|
|CLT |1347494|
|LAX |1086362|
|SEA |1026130|
|PHX |1021494|
|LAS |999980 |
|IAH |904123 |
|MCO |859424 |
|LGA |824201 |
|DTW |771676 |
|EWR |754431 |
|SFO |744383 |
|DCA |734667 |
|BOS |734643 |
|MSP |722020 |
|SLC |661231 |
|JFK |657671 |
|PHL |651792 |
|MIA |580061 |
|BNA |528700 |
|BWI |525397 |
|FLL |505521 |
+----+-------+
only showing top 25 rows

In [18]:
destination_state_summary = (
    eda_df.groupBy("DestState")
          .count()
          .orderBy(desc("count"))
)

destination_state_summary.show(60, False)

+---------+-------+
|DestState|count  |
+---------+-------+
|TX       |4409467|
|CA       |4089330|
|FL       |3456663|
|IL       |2320696|
|GA       |2078780|
|NY       |1988609|
|NC       |1913388|
|CO       |1898292|
|VA       |1543279|
|WA       |1235629|
|AZ       |1185274|
|NV       |1123332|
|PA       |1026139|
|MI       |1000308|
|TN       |834736 |
|NJ       |782122 |
|MN       |765503 |
|MA       |758893 |
|MO       |708043 |
|UT       |699676 |
|HI       |679451 |
|OH       |563206 |
|MD       |536098 |
|OR       |522236 |
|KY       |429538 |
|LA       |405412 |
|SC       |395455 |
|IN       |354186 |
|WI       |333446 |
|AK       |241013 |
|OK       |240687 |
|AL       |217386 |
|ID       |201687 |
|PR       |199535 |
|MT       |186364 |
|AR       |168324 |
|NE       |164882 |
|IA       |157590 |
|NM       |156335 |
|CT       |132438 |
|ND       |107417 |
|ME       |101408 |
|KS       |91923  |
|RI       |89453  |
|MS       |86864  |
|SD       |85227  |
|WY       |71222  |


In [19]:
month_summary = (
    eda_df.groupBy("Month")
          .count()
          .orderBy("Month")
)

month_summary.show()

+-----+-------+
|Month|  count|
+-----+-------+
|    1|3358992|
|    2|3141722|
|    3|3668894|
|    4|3246144|
|    5|3249065|
|    6|3352655|
|    7|3617203|
|    8|3590730|
|    9|3341566|
|   10|3525386|
|   11|3378386|
|   12|3439510|
+-----+-------+

In [20]:
year_summary = (
    eda_df.groupBy("Year")
          .count()
          .orderBy("Year")
)

year_summary.show()

+----+-------+
|Year|  count|
+----+-------+
|2020|5022397|
|2021|6311871|
|2022|7013508|
|2023|7278739|
|2024|7546968|
|2025|7736770|
+----+-------+

In [21]:
eda_df.groupBy(
    "Year",
    "Marketing_Airline_Network"
).count().orderBy(
    "Year",
    desc("count")
).show(200, False)

+----+-------------------------+-------+
|Year|Marketing_Airline_Network|count  |
+----+-------------------------+-------+
|2020|AA                       |1311462|
|2020|DL                       |1066908|
|2020|WN                       |961276 |
|2020|UA                       |885351 |
|2020|AS                       |282967 |
|2020|B6                       |144163 |
|2020|NK                       |135102 |
|2020|G4                       |98489  |
|2020|F9                       |91175  |
|2020|HA                       |45504  |
|2021|AA                       |1674371|
|2021|DL                       |1357322|
|2021|UA                       |1148340|
|2021|WN                       |1064640|
|2021|AS                       |359458 |
|2021|B6                       |202702 |
|2021|NK                       |191361 |
|2021|F9                       |137142 |
|2021|G4                       |115881 |
|2021|HA                       |60654  |
|2022|AA                       |1759257|
|2022|DL        

In [22]:
eda_df.groupBy(
    "Month",
    "Marketing_Airline_Network"
).count().orderBy(
    "Month",
    desc("count")
).show(200, False)

+-----+-------------------------+------+
|Month|Marketing_Airline_Network|count |
+-----+-------------------------+------+
|1    |AA                       |861868|
|1    |DL                       |710134|
|1    |UA                       |622812|
|1    |WN                       |601639|
|1    |AS                       |180986|
|1    |B6                       |115725|
|1    |NK                       |106068|
|1    |F9                       |73960 |
|1    |G4                       |48871 |
|1    |HA                       |36929 |
|2    |AA                       |807401|
|2    |DL                       |656395|
|2    |UA                       |587173|
|2    |WN                       |552140|
|2    |AS                       |168757|
|2    |B6                       |113215|
|2    |NK                       |100551|
|2    |F9                       |71383 |
|2    |G4                       |51728 |
|2    |HA                       |32979 |
|3    |AA                       |922680|
|3    |DL       

In [23]:
route_summary = (
    eda_df.groupBy(
        "Origin",
        "Dest"
    )
    .count()
    .orderBy(desc("count"))
)

route_summary.show(30, False)

+------+----+-----+
|Origin|Dest|count|
+------+----+-----+
|LAX   |SFO |65140|
|SFO   |LAX |65091|
|HNL   |OGG |60102|
|OGG   |HNL |60101|
|LAX   |LAS |57531|
|LAS   |LAX |57433|
|ORD   |LGA |56551|
|LGA   |ORD |56533|
|JFK   |LAX |53628|
|LAX   |JFK |53619|
|PHX   |DEN |49907|
|DEN   |PHX |49859|
|BOS   |DCA |49205|
|DCA   |BOS |49133|
|MCO   |ATL |47291|
|ATL   |MCO |47234|
|SEA   |PDX |46392|
|PDX   |SEA |46343|
|LAS   |DEN |44661|
|DEN   |LAS |44636|
|DEN   |SLC |43089|
|SLC   |DEN |43051|
|HNL   |LIH |42503|
|LIH   |HNL |42502|
|FLL   |ATL |42458|
|ATL   |FLL |42441|
|DEN   |LAX |42431|
|SEA   |LAX |42374|
|LAX   |SEA |42330|
|LAX   |DEN |42266|
+------+----+-----+
only showing top 30 rows

In [24]:
eda_df.groupBy(
    "OriginState",
    "DestState"
).count().orderBy(
    desc("count")
).show(30, False)

+-----------+---------+-------+
|OriginState|DestState|count  |
+-----------+---------+-------+
|CA         |CA       |1158052|
|TX         |TX       |1090599|
|HI         |HI       |398349 |
|TX         |CA       |377502 |
|CA         |TX       |377410 |
|FL         |TX       |367607 |
|TX         |FL       |367515 |
|FL         |NY       |341623 |
|NY         |FL       |341612 |
|FL         |GA       |334181 |
|GA         |FL       |334131 |
|CA         |NV       |327390 |
|NV         |CA       |327188 |
|WA         |CA       |286369 |
|CA         |WA       |286327 |
|AZ         |CA       |285721 |
|CA         |AZ       |285558 |
|FL         |NC       |255147 |
|NC         |FL       |255143 |
|CO         |CA       |249916 |
|CA         |CO       |249770 |
|TX         |CO       |216875 |
|CO         |TX       |216844 |
|FL         |IL       |201303 |
|IL         |FL       |201204 |
|NC         |NC       |198326 |
|CO         |CO       |181204 |
|NJ         |FL       |178645 |
|FL     